In [1]:
# Instalación de dependencias
!pip install datasets scikit-learn rank_bm25 pandas numpy

## **3.1. Preparación e indexación**
1. Cargar e indexar los documentos del corpus asignado.


In [2]:
# Carga de datos desde Hugging Face (con el codigo que proporciona la pagina)
from datasets import load_dataset

# Cargar documentos (corpus), consultas y juicios de relevancia (qrels)
corpus_ds = load_dataset("BeIR/nfcorpus", "corpus", split="corpus")
queries_ds = load_dataset("BeIR/nfcorpus", "queries", split="queries")
qrels_ds = load_dataset("BeIR/nfcorpus-qrels", split="test")

print(f"Total de documentos cargados: {len(corpus_ds)}")
print(f"Total de consultas de prueba cargadas: {len(queries_ds)}")
print(f"Total de qrels de prueba cargados: {len(qrels_ds)}")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

corpus/corpus-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.16MB            

corpus/corpus-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating corpus split:   0%|          | 0/3633 [00:00<?, ? examples/s]

queries/queries-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 80.9kB            

queries/queries-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (5 outputs left out)

Generating train split:   0%|          | 0/110575 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11385 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12334 [00:00<?, ? examples/s]

Total de documentos cargados: 3633
Total de consultas de prueba cargadas: 3237
Total de qrels de prueba cargados: 12334


In [4]:
# Diccionarios y listas para gestionar los documentos y sus IDs originales
documents = {}
doc_corpus_texts = []
doc_ids = []

for item in corpus_ds:
    doc_id = item["_id"]
    title = item.get("title", "")
    text = item.get("text", "")

    # Combinar título y texto
    combined_text = f"{title} {text}"

    documents[doc_id] = {
        "title": title,
        "text": text,
        "raw_text": combined_text
    }
    doc_corpus_texts.append(combined_text)
    doc_ids.append(doc_id)

print(f"Ejemplo de ID original del primer documento: {doc_ids[0]}")

Ejemplo de ID original del primer documento: MED-10


2. Aplicar y documentar las decisiones de preprocesamiento: tokenización, normalización, eliminación de stopwords y stemming o lematización.

In [5]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [7]:
# Descargar stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [8]:
def preprocess_text(text):
    if not text:
        return ""
    # Normalización: Convertir a minúsculas
    text = text.lower()

    # Tokenización y limpieza: Filtrar caracteres, eliminar puntuación y números
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()

    # Filtrado de stopwords y aplicación de stemming
    tokens = [stemmer.stem(word) for word in tokens if word not in stop_words and len(word) > 2]

    return " ".join(tokens)

# Aplicar el preprocesamiento a TODO el corpus
processed_corpus_texts = [preprocess_text(doc) for doc in doc_corpus_texts]

print("Texto Original (extracto):", doc_corpus_texts[0][:150])
print("Texto Preprocesado (extracto):", processed_corpus_texts[0][:150])

Texto Original (extracto): Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland Recent studies have suggested that statins, an established drug group in
Texto Preprocesado (extracto): statin use breast cancer surviv nationwid cohort studi finland recent studi suggest statin establish drug group prevent cardiovascular mortal could de


3. Construir un vocabulario y un índice invertido que permita recuperar documentos.

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi
import pandas as pd

# Construcción del índice TF-IDF (genera el vocabulario y la matriz de pesos)
tfidf_vectorizer = TfidfVectorizer(max_features=25000)
tfidf_matrix = tfidf_vectorizer.fit_transform(processed_corpus_texts)

# Construcción del índice BM25 (requiere el corpus tokenizado en listas de palabras)
tokenized_corpus = [doc.split() for doc in processed_corpus_texts]
bm25_index = BM25Okapi(tokenized_corpus)

# Obtener estadísticas del vocabulario construido
vocabulary_size = len(tfidf_vectorizer.get_feature_names_out())

# Muestra del vocabulario
feature_names = tfidf_vectorizer.get_feature_names_out()
print(f"Tamaño total del vocabulario generado: {len(feature_names)} términos únicos.\n")
print(f"Muestra de términos del vocabulario (del 100 al 115): {feature_names[100:115]}\n")

# Ejemplo de Índice Invertido (Término -> [IDs de documentos que lo contienen])
inverted_index_sample = {}
terms_to_show = ["statin", "cancer", "breast", "cardiovascular", "patient"]

for term in terms_to_show:
    if term in tfidf_vectorizer.vocabulary_:
        term_idx = tfidf_vectorizer.vocabulary_[term]
        # Buscar qué documentos contienen este término (peso TF-IDF > 0)
        docs_containing = tfidf_matrix.getcol(term_idx).nonzero()[0]
        # Mapear a los IDs originales del corpus
        original_doc_ids = [doc_ids[i] for i in docs_containing[:5]] # Mostramos hasta 5 de ejemplo
        inverted_index_sample[term] = {
            "frecuencia_en_documentos": len(docs_containing),
            "ejemplos_documentos_ids": original_doc_ids
        }

print("Evidencia del Índice Invertido")
for k, v in inverted_index_sample.items():
    print(f"Término: '{k}' -> Aparece en {v['frecuencia_en_documentos']} documentos. Ejemplos de IDs: {v['ejemplos_documentos_ids']}")

print(f"\nDimensiones de la matriz TF-IDF (Documentos x Términos): {tfidf_matrix.shape}")
print(f"Modelo BM25 inicializado con éxito para {len(tokenized_corpus)} documentos.")

Tamaño total del vocabulario generado: 16183 términos únicos.

Muestra de términos del vocabulario (del 100 al 115): ['acetochlor' 'acetolact' 'aceton' 'acetonitril' 'acetoxymethyl' 'acetyl'
 'acetylaspart' 'acetylcholin' 'acetylcystein' 'acetylcystin'
 'acetylgalactosamin' 'acetylmorphin' 'acetyltransferas' 'ach' 'achiev']

Evidencia del Índice Invertido
Término: 'statin' -> Aparece en 27 documentos. Ejemplos de IDs: ['MED-10', 'MED-14', 'MED-870', 'MED-1193', 'MED-1207']
Término: 'cancer' -> Aparece en 787 documentos. Ejemplos de IDs: ['MED-10', 'MED-14', 'MED-702', 'MED-712', 'MED-716']
Término: 'breast' -> Aparece en 281 documentos. Ejemplos de IDs: ['MED-10', 'MED-14', 'MED-666', 'MED-734', 'MED-753']
Término: 'cardiovascular' -> Aparece en 414 documentos. Ejemplos de IDs: ['MED-10', 'MED-329', 'MED-330', 'MED-332', 'MED-398']
Término: 'patient' -> Aparece en 781 documentos. Ejemplos de IDs: ['MED-10', 'MED-14', 'MED-301', 'MED-329', 'MED-330']

Dimensiones de la matriz TF-IDF (Do

Conservar la relación entre cada documento indexado y su identificador original.

In [14]:
# Actualizar para incluir el texto preprocesado
indexed_documents = {}

for idx, doc_id in enumerate(doc_ids):
    indexed_documents[idx] = {
        "document_id": doc_id,
        "title": documents[doc_id]["title"],
        "text": documents[doc_id]["text"],
        "processed_text": processed_corpus_texts[idx]
    }

# Verificar la consistencia con un ejemplo de mapeo bidireccional
sample_idx = 300
sample_original_id = indexed_documents[sample_idx]["document_id"]

print(f"Índice numérico interno: {sample_idx}")
print(f"Identificador original del corpus (necesario para los qrels): {sample_original_id}")
print(f"Título asociado: {indexed_documents[sample_idx]['title']}")

Índice numérico interno: 300
Identificador original del corpus (necesario para los qrels): MED-1137
Título asociado: Diet and risk of kidney stones in the Oxford cohort of the European Prospective Investigation into Cancer and Nutrition (EPIC).


## **3.2. Modelos de recuperación**

Función de Búsqueda con TF-IDF y Similitud Coseno

In [16]:
from sklearn.metrics.pairwise import cosine_similarity

def search_tfidf(query_text, top_k=10):
    # Preprocesar la consulta del usuario usando la misma función anterior
    query_processed = preprocess_text(query_text)

    # Vectorizar la consulta con el modelo TF-IDF ya entrenado
    query_vec = tfidf_vectorizer.transform([query_processed])

    # Calcular la similitud coseno entre la consulta y todos los documentos del corpus
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # Obtener los índices de los documentos ordenados de mayor a menor puntuación
    top_indices = scores.argsort()[::-1][:top_k]

    # Estructurar los resultados con el adecuado formato
    results = []
    for rank, idx in enumerate(top_indices, start=1):
        d_id = doc_ids[idx]
        results.append({
            "rank": rank,
            "document_id": d_id,
            "score": float(scores[idx]),
            "title": indexed_documents[idx]["title"],
            "snippet": indexed_documents[idx]["text"][:150] + "..."
        })
    return results

print("Función de búsqueda TF-IDF definida correctamente.")

Función de búsqueda TF-IDF definida correctamente.


Función de Búsqueda con BM25

In [17]:
def search_bm25(query_text, top_k=10):
    # Preprocesar y tokenizar la consulta
    query_processed = preprocess_text(query_text)
    tokenized_query = query_processed.split()

    # Calcular las puntuaciones de relevancia con el índice BM25
    scores = bm25_index.get_scores(tokenized_query)

    # Obtener los índices ordenados de mayor a menor puntaje
    top_indices = scores.argsort()[::-1][:top_k]

    # Estructurar los resultados en el adecuado formato
    results = []
    for rank, idx in enumerate(top_indices, start=1):
        d_id = doc_ids[idx]
        results.append({
            "rank": rank,
            "document_id": d_id,
            "score": float(scores[idx]),
            "title": indexed_documents[idx]["title"],
            "snippet": indexed_documents[idx]["text"][:150] + "..."
        })
    return results

print("Función de búsqueda BM25 definida correctamente.")

Función de búsqueda BM25 definida correctamente.


Prueba y comparación preliminar de ambos modelos

In [19]:
sample_query = "statin use and breast cancer survival"

print(f"Consulta de prueba: '{sample_query}'\n")

print("___Ranking obtenido con TF-IDF___ ")
tfidf_res = search_tfidf(sample_query, top_k=3)
for res in tfidf_res:
    print(f"[{res['rank']}] ID: {res['document_id']} | Score: {res['score']:.4f}")
    print(f"    Título: {res['title']}\n")

print("///Ranking obtenido con BM25///")
bm25_res = search_bm25(sample_query, top_k=3)
for res in bm25_res:
    print(f"[{res['rank']}] ID: {res['document_id']} | Score: {res['score']:.4f}")
    print(f"    Título: {res['title']}\n")

Consulta de prueba: 'statin use and breast cancer survival'

___Ranking obtenido con TF-IDF___ 
[1] ID: MED-10 | Score: 0.7614
    Título: Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland

[2] ID: MED-14 | Score: 0.6336
    Título: Statin use after diagnosis of breast cancer and survival: a population-based cohort study.

[3] ID: MED-2429 | Score: 0.6061
    Título: Statin use and risk of breast cancer: a meta-analysis of observational studies.

///Ranking obtenido con BM25///
[1] ID: MED-10 | Score: 24.0965
    Título: Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland

[2] ID: MED-14 | Score: 21.3858
    Título: Statin use after diagnosis of breast cancer and survival: a population-based cohort study.

[3] ID: MED-2429 | Score: 17.9765
    Título: Statin use and risk of breast cancer: a meta-analysis of observational studies.



## **3.3. Servicio web**

In [28]:
!pip install fastapi uvicorn pyngrok

In [31]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn
import nest_asyncio
from pyngrok import ngrok
import threading
import socket

app = FastAPI(title="Examen IR - Sistema de Recuperación de Información", version="1.0")

# Definir el esquema de la solicitud POST esperada
class QueryRequest(BaseModel):
    query: str
    model: str  # "bm25" o "tfidf"
    top_k: int = 10

@app.get("/health")
def health_check():
    return {"status": "ok", "message": "El servicio web está activo y operativo."}

@app.post("/search")
def search_documents(request: QueryRequest):
    query_text = request.query.strip()
    model_name = request.model.lower().strip()
    top_k = request.top_k

    if not query_text:
        raise HTTPException(status_code=400, detail="El campo 'query' no puede estar vacío.")

    if model_name == "tfidf":
        results = search_tfidf(query_text, top_k=top_k)
    elif model_name == "bm25":
        results = search_bm25(query_text, top_k=top_k)
    else:
        raise HTTPException(status_code=400, detail="Modelo no válido. Utilice 'bm25' o 'tfidf'.")

    return {
        "model": model_name,
        "results": results
    }

# Configuración para correr Uvicorn dentro de Jupyter Notebooks
nest_asyncio.apply()

# Función auxiliar para comprobar si el puerto ya está ocupado
def is_port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('127.0.0.1', port)) == 0

# Lanzar el servidor local solo si el puerto está libre
if not is_port_in_use(8000):
    def run_server():
        uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

    server_thread = threading.Thread(target=run_server, daemon=True)
    server_thread.start()
    print(" * Backend local de FastAPI iniciado exitosamente en el puerto 8000.")
else:
    print(" * El servidor FastAPI ya se encuentra activo en segundo plano (puerto 8000).")

# ==========================================================================
# CONFIGURACIÓN OBLIGATORIA DE NGROK:
# ==========================================================================
NGROK_TOKEN = "ngrok config add-authtoken 3MI0baSNLpFhfTKdnOfN1OSG9Al_biHWEi1XqMWBHrd94zve"

# Limpiar el token automáticamente en caso de haber copiado el comando de terminal completo
clean_token = NGROK_TOKEN.replace("ngrok config add-authtoken", "").strip()

# Matar túneles activos de ejecuciones previas para evitar conflictos
try:
    ngrok.kill()
except Exception:
    pass

if not clean_token or clean_token == "ngrok config add-authtoken 3MI0baSNLpFhfTKdnOfN1OSG9Al_biHWEi1XqMWBHrd94zve":
    print("\n⚠️ ADVERTENCIA: Debes ingresar un token de ngrok válido en la variable 'NGROK_TOKEN'.")
    print("Regístrate gratis en https://dashboard.ngrok.com/signup y copia tu authtoken.")
else:
    try:
        ngrok.set_auth_token(clean_token)
        public_url = ngrok.connect(8000).public_url
        print(f"\n * URL Pública de tu Servicio Web: {public_url}")
        print(" * Copia esta URL para entregarla como requerimiento del examen.")
    except Exception as e:
        print(f"\n❌ Error al conectar a ngrok: {e}")
        print("Por favor, revisa que tu token sea el correcto.")

 * El servidor FastAPI ya se encuentra activo en segundo plano (puerto 8000).

 * URL Pública de tu Servicio Web: https://smock-navigate-subsystem.ngrok-free.dev
 * Copia esta URL para entregarla como requerimiento del examen.


# **4. Evaluación de la recuperación**

Cada estudiante debe evaluar ambos modelos con consultas de la partición test y sus qrels. Presentará, como mínimo:

* Precision@10.
* Recall@10.
* NDCG@10.
* Una comparación de los resultados de TF-IDF y BM25.




In [20]:
import numpy as np

# Agrupar los qrels de prueba por query_id y corpus-id con sus scores de relevancia
qrels_dict = {}
for row in qrels_ds:
    q_id = str(row["query-id"])
    d_id = str(row["corpus-id"])
    rel = int(row["score"])
    if q_id not in qrels_dict:
        qrels_dict[q_id] = {}
    qrels_dict[q_id][d_id] = rel

# Mapear las consultas de texto por su ID oficial
queries_dict = {str(item["_id"]): item["text"] for item in queries_ds}

In [21]:
def evaluate_retrieval_model(model_type, sample_size=50):
    precisions, recalls, ndcgs = [], [], []
    evaluated_count = 0

    for q_id, relevant_docs in qrels_dict.items():
        if q_id not in queries_dict or evaluated_count >= sample_size:
            continue
        query_text = queries_dict[q_id]

        # Obtener los top 10 resultados según el modelo evaluado
        if model_type == "tfidf":
            res = search_tfidf(query_text, top_k=10)
        else:
            res = search_bm25(query_text, top_k=10)

        retrieved_ids = [r["document_id"] for r in res]

        # Calcular Precision@10
        rel_retrieved = sum(1 for d in retrieved_ids if d in relevant_docs and relevant_docs[d] > 0)
        p_10 = rel_retrieved / 10.0

        # Calcular Recall@10
        total_relevant = sum(1 for d in relevant_docs.values() if d > 0)
        r_10 = rel_retrieved / total_relevant if total_relevant > 0 else 0.0

        # Calcular NDCG@10
        dcg = sum((relevant_docs.get(d, 0)) / np.log2(idx + 2) for idx, d in enumerate(retrieved_ids))
        ideal_sorted = sorted(relevant_docs.values(), reverse=True)[:10]
        idcg = sum(score / np.log2(idx + 2) for idx, score in enumerate(ideal_sorted))
        ndcg = dcg / idcg if idcg > 0 else 0.0

        precisions.append(p_10)
        recalls.append(r_10)
        ndcgs.append(ndcg)
        evaluated_count += 1

    return np.mean(precisions), np.mean(recalls), np.mean(ndcgs)

In [22]:
print("Calculando métricas para TF-IDF...")
tfidf_p10, tfidf_r10, tfidf_ndcg = evaluate_retrieval_model("tfidf", sample_size=50)

print("Calculando métricas para BM25...")
bm25_p10, bm25_r10, bm25_ndcg = evaluate_retrieval_model("bm25", sample_size=50)

# Construir tabla comparativa de resultados
evaluation_df = pd.DataFrame({
    "Modelo de Recuperación": ["TF-IDF (Coseno)", "BM25"],
    "Precision@10": [tfidf_p10, bm25_p10],
    "Recall@10": [tfidf_r10, bm25_r10],
    "NDCG@10": [tfidf_ndcg, bm25_ndcg]
})

print("\n--- Resultados de la Evaluación ---")
display(evaluation_df)

Calculando métricas para TF-IDF...
Calculando métricas para BM25...

--- Resultados de la Evaluación ---


,Modelo de Recuperación,Precision@10,Recall@10,NDCG@10
0,TF-IDF (Coseno),0.210,0.142287,0.271135
1,BM25,0.248,0.158178,0.327961
